In [9]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
%pip install -q -U torch transformers accelerate huggingface_hub safetensors bitsandbytes
# https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct

from transformers import AutoTokenizer
import torch

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DTYPE = (
    torch.float16
    if DEVICE == "cuda"
    else torch.float32
)

print("Device :", DEVICE)
print("Dtype  :", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

print("Tokenizer chargé.")



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Device : cpu
Dtype  : torch.float32
Tokenizer chargé.


In [11]:
# https://huggingface.co/docs/transformers/quantization/bitsandbytes?bnb=4-bit
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


model_quantized = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

model_quantized.eval()

print("Modèle quantifié en 4 bits chargé.")


Modèle quantifié en 4 bits chargé.


In [12]:
PATH = "./models/quantized_model"

model_quantized.save_pretrained(
    PATH,
    safe_serialization=True
)

tokenizer.save_pretrained(
    PATH
)

print(
    f"Modèle sauvegardé dans : {PATH}"
)

Modèle sauvegardé dans : ./models/quantized_model
